# Feature distributions: raw training data vs. SMOTE-ENN

This notebook compares only the **80% raw training split** used by the resampling pipeline with its saved **post-SMOTE-ENN training dataset**. The untouched 20% raw holdout is deliberately excluded.

Each feature is plotted side by side with identical x- and y-scales, so shifts caused by SMOTE-ENN are visible. Figures and summary tables are logged to a dedicated Weights & Biases run.

## 1. Setup and load the exact comparison datasets

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import wandb
import yaml
from scipy.stats import ks_2samp, wasserstein_distance
from sklearn.model_selection import train_test_split

REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / 'configs').is_dir():
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / 'configs').is_dir():
    raise FileNotFoundError('Run this notebook from inside the repository.')
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.data.load import load_raw

sns.set_theme(style='whitegrid', context='notebook')
cfg = yaml.safe_load((REPO_ROOT / 'configs' / 'resample_smoteenn.yaml').read_text())
data_cfg = cfg['data']
TARGET = data_cfg['target_column']
SEED = int(cfg['random_seed'])
TEST_SIZE = float(data_cfg['test_size'])

raw_df = load_raw(REPO_ROOT / data_cfg['input_path'], target_column=TARGET)
raw_train_df, _ = train_test_split(
    raw_df, test_size=TEST_SIZE, random_state=SEED, stratify=raw_df[TARGET]
)
raw_train_df = raw_train_df.reset_index(drop=True)
resampled_df = pd.read_csv(REPO_ROOT / data_cfg['output_path'])

FEATURES = [column for column in raw_train_df.columns if column != TARGET]
assert set(raw_train_df.columns) == set(resampled_df.columns), 'The two datasets have different schemas.'
assert len(raw_train_df) + round(len(raw_df) * TEST_SIZE) == len(raw_df), 'Unexpected train/holdout split.'

print(f'Raw data: {raw_df.shape[0]:,} rows')
print(f'Raw training split (80%): {raw_train_df.shape[0]:,} rows')
print(f'Raw holdout (20%, excluded): {len(raw_df) - len(raw_train_df):,} rows')
print(f'Post-SMOTE-ENN training data: {resampled_df.shape[0]:,} rows')

## 2. Start a W&B distribution-audit run

Set `WANDB_MODE = 'disabled'` if you only want local inline plots.

In [ ]:
WANDB_MODE = cfg.get('wandb', {}).get('mode', 'online')
run = wandb.init(
    project=cfg.get('wandb', {}).get('project', 'diabetes-brfss-ml'),
    entity=cfg.get('wandb', {}).get('entity'),
    name='mahdi-feature-distribution-audit',
    job_type='data-audit',
    tags=['mahdi', 'data-audit', 'smote-enn', 'feature-distributions'],
    mode=WANDB_MODE,
    config={
        'random_seed': SEED,
        'test_size': TEST_SIZE,
        'comparison': 'raw 80% training split vs post-SMOTE-ENN training data',
        'raw_train_rows': len(raw_train_df),
        'resampled_train_rows': len(resampled_df),
    },
)
print(f'W&B run: {run.url if run.url else WANDB_MODE}')

## 3. Target balance check

This is shown separately from feature distributions because it is the label altered directly by the resampling strategy.

In [ ]:
class_rates = pd.DataFrame({
    'raw 80% training split': raw_train_df[TARGET].value_counts(normalize=True).sort_index(),
    'post-SMOTE-ENN training': resampled_df[TARGET].value_counts(normalize=True).sort_index(),
}).rename_axis('class').reset_index().melt('class', var_name='dataset', value_name='share')

fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(data=class_rates, x='class', y='share', hue='dataset', ax=ax)
ax.set(title='Target-class balance: before vs. after SMOTE-ENN', xlabel='Diabetes class', ylabel='Share of rows', ylim=(0, 1))
ax.legend(title='Dataset')
fig.tight_layout()
run.log({'distribution_audit/target_class_balance': wandb.Image(fig), 'distribution_audit/target_class_balance_table': wandb.Table(dataframe=class_rates)})
plt.show()
plt.close(fig)

## 4. Feature distributions, side by side

Bar charts are used for binary features. All other features use matched-bin, normalized histograms; fractional bars reveal any non-integer values introduced by SMOTE in originally discrete features.

In [ ]:
def plot_feature_comparison(feature):
    before = raw_train_df[feature].dropna().to_numpy()
    after = resampled_df[feature].dropna().to_numpy()
    combined = np.concatenate([before, after])
    raw_unique = np.unique(before)
    is_binary = len(raw_unique) <= 2 and set(raw_unique).issubset({0, 1})

    fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharex=True, sharey=True)
    if is_binary:
        categories = np.array([0, 1])
        before_share = pd.Series(before).value_counts(normalize=True).reindex(categories, fill_value=0)
        after_share = pd.Series(after).value_counts(normalize=True).reindex(categories, fill_value=0)
        axes[0].bar(categories, before_share, color='#4C78A8', width=0.65)
        axes[1].bar(categories, after_share, color='#F58518', width=0.65)
        for ax in axes:
            ax.set_xticks(categories)
    else:
        if len(raw_unique) <= 20:
            lo, hi = combined.min(), combined.max()
            bins = np.linspace(lo - 0.5, hi + 0.5, max(8, min(41, int((hi - lo) * 3) + 2)))
        else:
            bins = np.histogram_bin_edges(combined, bins=40)
        axes[0].hist(before, bins=bins, weights=np.full(len(before), 1 / len(before)), color='#4C78A8', alpha=0.9)
        axes[1].hist(after, bins=bins, weights=np.full(len(after), 1 / len(after)), color='#F58518', alpha=0.9)

    axes[0].set_title('Raw 80% training split (before)')
    axes[1].set_title('Post-SMOTE-ENN training (after)')
    for ax in axes:
        ax.set_xlabel(feature)
        ax.set_ylabel('Share of rows')
        ax.grid(axis='y', alpha=0.25)
    fig.suptitle(f'{feature}: feature distribution before vs. after resampling', y=1.03, fontsize=13)
    fig.tight_layout()
    return fig

for feature in FEATURES:
    fig = plot_feature_comparison(feature)
    run.log({f'distribution_audit/features/{feature}': wandb.Image(fig)})
    plt.show()
    plt.close(fig)

## 5. Numerical distribution-shift summary

The Kolmogorov-Smirnov statistic measures the largest gap between the two empirical distributions (0 means identical); Wasserstein distance measures how far values move on the feature's native scale.

In [ ]:
summary_rows = []
for feature in FEATURES:
    before = raw_train_df[feature].dropna().to_numpy()
    after = resampled_df[feature].dropna().to_numpy()
    ks = ks_2samp(before, after)
    summary_rows.append({
        'feature': feature,
        'raw_mean': before.mean(),
        'resampled_mean': after.mean(),
        'mean_shift': after.mean() - before.mean(),
        'raw_std': before.std(),
        'resampled_std': after.std(),
        'ks_statistic': ks.statistic,
        'ks_pvalue': ks.pvalue,
        'wasserstein_distance': wasserstein_distance(before, after),
        'raw_unique_values': len(np.unique(before)),
        'resampled_unique_values': len(np.unique(after)),
        'resampled_fractional_share': float(np.mean(~np.isclose(after, np.round(after)))),
    })

shift_summary = pd.DataFrame(summary_rows).sort_values('ks_statistic', ascending=False).reset_index(drop=True)
display(shift_summary.style.format({
    'raw_mean': '{:.3f}', 'resampled_mean': '{:.3f}', 'mean_shift': '{:+.3f}',
    'raw_std': '{:.3f}', 'resampled_std': '{:.3f}', 'ks_statistic': '{:.4f}',
    'ks_pvalue': '{:.2e}', 'wasserstein_distance': '{:.4f}', 'resampled_fractional_share': '{:.2%}',
}))
run.log({'distribution_audit/feature_shift_summary': wandb.Table(dataframe=shift_summary)})
run.summary['distribution_audit/largest_ks_feature'] = shift_summary.iloc[0]['feature']
run.summary['distribution_audit/largest_ks_statistic'] = float(shift_summary.iloc[0]['ks_statistic'])

In [ ]:
resampled_df.genhlth.value_counts()

In [ ]:
raw_df.genhlth.value_counts()

## 6. Finish the W&B run

In [ ]:
run.finish()
print('Distribution audit logged to W&B.')